# w16_gpu1_ppt_mainline_corrected_h_and_ego_poly
Inputs: `shortpose-wave1-private-inputs` | Accelerator: `GPU T4 x2`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir() and torch.cuda.is_available(), 'Kaggle GPU required'
n_gpu = torch.cuda.device_count()
print('GPUs:', n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)], flush=True)
REV = 'c005d36c19d31972eaadca067ffcfc4c885f9e1c'
with tempfile.TemporaryDirectory(prefix='w16_g1_') as ts:
    tmp = Path(ts)
    repo, meta_dir = tmp / 'repo', tmp / 'pinns_meta'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    base_env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=base_env, check=True)
    d1 = WORK / 'ppt_mainline_o2_m0_m3_corrected_v3.zip'
    d2 = WORK / 'pinns_short_disp_hardq_anchor_corrected_v2.zip'
    e0 = {**base_env, 'CUDA_VISIBLE_DEVICES': '0'}
    e1 = {**base_env, 'CUDA_VISIBLE_DEVICES': '1' if n_gpu > 1 else '0'}
    p1 = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.run_ppt_mainline_o2_m0_m3_and_fixed_conformal', str(meta_dir), str(pose_npz), str(joint_bin), str(map_bin), str(d1)], cwd=repo, env=e0)
    p2 = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.run_pinns_short_disp_hardq_anchor_5seed', str(meta_dir), str(pose_npz), str(d2)], cwd=repo, env=e1)
    rc1, rc2 = p1.wait(), p2.wait()
    assert rc1 == 0 and rc2 == 0, f'Subprocess failed: rc1={rc1}, rc2={rc2}'
    for d in (d1, d2):
        print('DONE:', d, d.stat().st_size, hashlib.sha256(d.read_bytes()).hexdigest(), flush=True)
